In [0]:
from pyspark.sql.functions import col, to_date, date_format, udf
from pyspark.sql.types import DateType
from datetime import datetime

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df = spark.table("bronze.dwp_claimant_count")

def parse_month_yearr(s):
    if s is None:
        return None
    try:
        return datetime.strptime(s, "%B %Y").date()
    except:
        return None

parse_date_udf = udf(parse_month_yearr, DateType())

df_silver = (
    df
    .withColumn("report_date", parse_date_udf(col("DATE_NAME")))
    .withColumn("year_month", date_format(col("report_date").cast("timestamp"), "yyyy-MM"))
    .withColumnRenamed("GEOGRAPHY_NAME", "council_name")
    .withColumnRenamed("GEOGRAPHY_CODE", "council_area_code")
    .withColumn("claimant_count", col("OBS_VALUE").cast("long"))
    .select("report_date", "year_month", "council_area_code", "council_name", "claimant_count")
)

df_silver.show(5)
df_silver.printSchema()

In [0]:
(
    df_silver.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dwp_claimant_count")
)

In [0]:
# silver.dwp_claimant_count
# 3,840 rows -- 32 Scottish council areas x 120 months
# April 2016 to March 2026
# report_date (date), year_month (string), council_area_code (string),
# council_name (string), claimant_count (long)